# ⚡ WavLM Extraction — PyTorch 2.2.2 + CUDA 11.8
**Fix:** Install PyTorch 2.2.2 with CUDA 11.8 (should support P100 sm_60)


In [ ]:
# Cell 1: Install PyTorch 2.2.2 + CUDA 11.8
import subprocess, os, sys

print('=== Installing PyTorch 2.2.2 + CUDA 11.8 ===')

# Uninstall current
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torch', 'torchvision', 'torchaudio', '-y', '-q'], capture_output=True)

# Install PyTorch 2.2.2 with CUDA 11.8
result = subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch==2.2.2',
    'torchvision==0.17.2',
    'torchaudio==2.2.2',
    '--index-url',
    'https://download.pytorch.org/whl/cu118'
], capture_output=True, text=True, timeout=600)

print('Return code:', result.returncode)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:] if result.stderr else 'none')

# Test
try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'CUDA: {torch.version.cuda}')
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        x = torch.randn(100, 100).cuda()
        y = x @ x
        print('GPU: SUCCESS')
except Exception as e:
    print(f'Import/GPU test failed: {e}')
    print('Installing default PyTorch...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'], capture_output=True)
    import torch
    print(f'Default PyTorch: {torch.__version__}')

In [ ]:
# Cell 2: Load WavLM
try:
    import torch
    from transformers import AutoModel
    
    print('Loading WavLM...')
    wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
    if torch.cuda.is_available():
        wavlm = wavlm.cuda()
        wavlm.eval()
        print('WavLM loaded on GPU')
    else:
        wavlm.eval()
        print('WavLM loaded on CPU')
except Exception as e:
    print(f'Failed to load WavLM: {e}')

In [ ]:
# Cell 3: Setup + Download
import subprocess, os

os.makedirs('/kaggle/working/audio', exist_ok=True)
os.makedirs('/kaggle/working/features', exist_ok=True)

subprocess.run(['pip', 'install', 'yt-dlp', '-q'], capture_output=True)

VIDEO_IDS = ['q112mLKiUCw', 'J9HLFJgUCW0', 'LWYfo_8t5WQ', 'oiRyNnyG698', 'KVMbGry8AgM']

def download_one(vid):
    out = f'/kaggle/working/audio/{vid}.wav'
    if os.path.exists(out): return out
    cmd = ['yt-dlp', '-f', 'bestaudio[ext=m4a]', '--extract-audio', '--audio-format', 'wav',
            '-o', f'/kaggle/working/audio/{vid}.%(ext)s',
            f'https://www.youtube.com/watch?v={vid}', '--no-playlist', '--quiet', '--socket-timeout', '90']
    try:
        subprocess.run(cmd, capture_output=True, timeout=180)
        for ext in ['m4a', 'webm', 'mp4']:
            tmp = f'/kaggle/working/audio/{vid}.{ext}'
            if os.path.exists(tmp) and tmp != out:
                os.rename(tmp, out)
        return out if os.path.exists(out) else None
    except: return None

print(f'Downloading {len(VIDEO_IDS)} videos...')
for vid in VIDEO_IDS:
    path = download_one(vid)
    print(f'  {vid}: {"OK" if path else "FAIL"}')

In [ ]:
# Cell 4: Extract features
import numpy as np, librosa, torch, time

def prosody23(y, sr):
    f = []
    try:
        f0, v, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        fc = f0[~np.isnan(f0)]; v = v[~np.isnan(f0)]
        f = [np.mean(fc) if len(fc)>0 else 0, np.std(fc) if len(fc)>0 else 0,
              np.max(fc) if len(fc)>0 else 0, np.min(fc) if len(fc)>0 else 0,
              np.mean(v) if len(v)>0 else 0]
    except: f = [0]*5
    hop=512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms),np.std(rms),np.max(rms),np.min(rms),np.max(rms)-np.min(rms)])
    dur=len(y)/sr; sr_r=dur/(np.sum(rms>np.mean(rms))+1)
    f.extend([dur, sr_r])
    try:
        sc=librosa.feature.spectral_centroid(y=y,sr=sr,hop_length=hop)[0]
        sb=librosa.feature.spectral_bandwidth(y=y,sr=sr,hop_length=hop)[0]
        sf=librosa.feature.spectral_flatness(y=y,hop_length=hop)[0]
        z=librosa.feature.zero_crossing_rate(y,hop_length=hop)[0]
        f.extend([np.mean(sc),np.mean(sb),np.mean(sf),np.mean(z),np.std(z)])
    except: f.extend([0]*5)
    try:
        yh,_=librosa.effects.hpss(y)
        hnr=np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr,np.mean(np.abs(y)),np.std(y),np.max(np.abs(y)),0,0])
    except: f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def extract(audio_path, vid):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        cs = 16000*5; n = len(y)//cs
        if n == 0: return None
        feats = []
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        for i in range(n):
            ch = y[i*cs:(i+1)*cs]
            t = torch.tensor(ch).unsqueeze(0).to(dev)
            with torch.no_grad():
                r = wavlm(t).last_hidden_state.mean(dim=2).squeeze().cpu().numpy()
            p = prosody23(ch, 16000)
            feats.append(np.concatenate([r, p]))
        return np.array(feats)
    except Exception as e:
        print(f'  Error: {e}')
        return None

t0 = time.time()
for vid in VIDEO_IDS:
    ap = f'/kaggle/working/audio/{vid}.wav'
    if not os.path.exists(ap): continue
    f = extract(ap, vid)
    if f is not None:
        np.save(f'/kaggle/working/features/{vid}_features.npy', f)
        print(f'{vid}: {f.shape} ({time.time()-t0:.0f}s)')
    else:
        print(f'{vid}: FAILED')

In [ ]:
# Cell 5: Done
import os
fs = [f for f in os.listdir('/kaggle/working/features') if f.endswith('.npy')]
print(f'\nExtracted: {len(fs)} videos')
print('Done!')